# ECLIPSE Stronger Frozen Backbone - Swin-T Colab Version

This is a separate no-TTA notebook for testing the stronger frozen-backbone idea.

Run it top-to-bottom in VS Code while connected to a Google Colab runtime. The flow is:

1. Clone and build ECLIPSE/Detectron2.
2. Prepare ADE20K panoptic data.
3. Download and convert the official Swin-T ImageNet checkpoint.
4. Train the 100-class base step once with a Swin-T backbone.
5. Freeze the base model and train the continual visual prompts for 100-50, 100-10, and 100-5.
6. Evaluate with normal single-scale inference only, no TTA.
7. Plot original article all-PQ vs the final Swin-T results parsed from your logs.

Default training now uses a stronger Colab A100 schedule instead of the paper-length schedule: 75k base iterations, 12k for 100-50, 4k per 100-10 step, and 3k per 100-5 step. Cell 8.5 saves/restores checkpoints through Google Drive so reruns can skip already trained stages.


In [ ]:
# Cell 0 - Mount Drive and clone ECLIPSE cleanly

from google.colab import drive
drive.mount('/content/drive')

%cd /content
!rm -rf ECLIPSE_STRONG_BACKBONE
!git clone https://github.com/clovaai/ECLIPSE.git ECLIPSE_STRONG_BACKBONE

%cd /content/ECLIPSE_STRONG_BACKBONE
!pwd
!ls


In [ ]:

# Cell 1 — Install a compatible Torch/CUDA stack

%cd /content/ECLIPSE_STRONG_BACKBONE

# Current Colab system CUDA is usually 12.8.
# We pin PyTorch to CUDA 12.8, not CUDA 13.0, to avoid Detectron2 compile mismatch.
!python -m pip uninstall -y torch torchvision torchaudio detectron2

!python -m pip install -U pip setuptools wheel ninja cython

!python -m pip install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu128

!nvcc --version

import torch

print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

assert torch.version.cuda == "12.8", "Torch CUDA is not 12.8. Stop here."


In [ ]:
# Cell 2 — Install Python dependencies without upgrading Torch

%cd /content/ECLIPSE_STRONG_BACKBONE

!python -m pip install -U pip setuptools wheel ninja cython
!python -m pip install "numpy<2" "cython<3" pycocotools

# Install common Detectron2/ECLIPSE dependencies.
!python -m pip install \
    fvcore iopath omegaconf hydra-core cloudpickle termcolor yacs tabulate tqdm tensorboard \
    scipy scikit-image matplotlib pillow opencv-python-headless einops timm regex ftfy \
    git+https://github.com/cocodataset/panopticapi.git \
    git+https://github.com/mcordts/cityscapesScripts.git

# If ECLIPSE has requirements.txt, install it but ignore packages that can break the environment.
from pathlib import Path

req = Path("requirements.txt")

if req.exists():
    banned = ("torch", "torchvision", "torchaudio", "detectron2", "numpy", "opencv", "mmcv")
    clean_lines = []

    for line in req.read_text().splitlines():
        s = line.strip()

        if not s or s.startswith("#"):
            continue

        if s.lower().startswith(banned):
            print("Skipping requirement:", s)
            continue

        clean_lines.append(s)

    Path("/content/eclipse_requirements_clean.txt").write_text("\n".join(clean_lines))

    print("Installing cleaned requirements:")
    print(Path("/content/eclipse_requirements_clean.txt").read_text())

    !python -m pip install -r /content/eclipse_requirements_clean.txt

# Verify that Torch did not get upgraded.
import torch

print("Torch after dependencies:", torch.__version__)
print("Torch CUDA after dependencies:", torch.version.cuda)

assert torch.version.cuda == "12.8", "Torch changed. Stop and rerun from Cell 1."


In [ ]:
# Cell 3 — Build and verify Detectron2 from source

%cd /content
!rm -rf detectron2_src
!git clone https://github.com/facebookresearch/detectron2.git detectron2_src

%cd /content/detectron2_src

!rm -rf build
!find . -name "*.so" -delete

!FORCE_CUDA=1 python setup.py build_ext --inplace

!python -m pip install -e . --no-build-isolation

%cd /content/ECLIPSE_STRONG_BACKBONE

%env PYTHONPATH=/content/detectron2_src:/content/ECLIPSE_STRONG_BACKBONE

import detectron2
print("Detectron2:", detectron2.__file__)

from detectron2 import _C
print("Detectron2 _C OK")


In [ ]:
# Diagnose Cell 3 failure

%cd /content/ECLIPSE_STRONG_BACKBONE

import sys
from pathlib import Path

print("Python:", sys.executable)
print("detectron2_src folder exists:", Path("/content/detectron2_src").exists())
print("detectron2 package folder exists:", Path("/content/detectron2_src/detectron2").exists())

print("\nCompiled .so files inside detectron2_src:")
!find /content/detectron2_src -name "*.so" | head -20 || true

print("\nPip detectron2 status:")
!python -m pip show detectron2 || true

print("\nTry importing with forced sys.path:")
if "/content/detectron2_src" not in sys.path:
    sys.path.insert(0, "/content/detectron2_src")

try:
    import detectron2
    print("Detectron2 imported from:", detectron2.__file__)

    from detectron2 import _C
    print("Detectron2 _C OK")
except Exception as e:
    print("IMPORT FAILED:")
    print(type(e).__name__, e)


In [ ]:
# Cell 3.5 — Force notebook and shell to see Detectron2

%cd /content/ECLIPSE_STRONG_BACKBONE

import sys

if "/content/detectron2_src" not in sys.path:
    sys.path.insert(0, "/content/detectron2_src")

%env PYTHONPATH=/content/detectron2_src:/content/ECLIPSE_STRONG_BACKBONE

import detectron2
print("Detectron2:", detectron2.__file__)

from detectron2 import _C
print("Detectron2 _C OK")


In [ ]:
# Cell 4 — Download and prepare ADE20K dataset

%cd /content/ECLIPSE_STRONG_BACKBONE

from pathlib import Path

Path("datasets").mkdir(exist_ok=True)

# Main ADE20K challenge data
%cd /content/ECLIPSE_STRONG_BACKBONE/datasets

if not Path("ADEChallengeData2016").exists():
    !wget -q http://data.csail.mit.edu/places/ADEchallenge/ADEChallengeData2016.zip
    !unzip -q ADEChallengeData2016.zip
    !rm ADEChallengeData2016.zip
else:
    print("ADEChallengeData2016 already exists, skipping main download.")

# Instance annotations
%cd /content/ECLIPSE_STRONG_BACKBONE/datasets/ADEChallengeData2016

if not Path("annotations_instance").exists():
    !wget -q http://sceneparsing.csail.mit.edu/data/ChallengeData2017/annotations_instance.tar
    !tar -xf annotations_instance.tar
    !rm annotations_instance.tar
else:
    print("annotations_instance already exists, skipping instance download.")

# Dataset preparation scripts
%cd /content/ECLIPSE_STRONG_BACKBONE

!python datasets/prepare_ade20k_sem_seg.py
!python datasets/prepare_ade20k_pan_seg.py
!python datasets/prepare_ade20k_ins_seg.py

print("ADE20K preparation finished.")


In [ ]:
# Cell 5 - Download and convert the Swin-T backbone checkpoint

%cd /content/ECLIPSE_STRONG_BACKBONE

from pathlib import Path

# Default: Swin-Tiny. You can change this to "swin_small" or "swin_large" before running
# if your Colab GPU and time budget can handle it.
BACKBONE_VARIANT = "swin_tiny"

BACKBONE_OPTIONS = {
    "swin_tiny": {
        "tag": "swin_t",
        "pth": "swin_tiny_patch4_window7_224.pth",
        "pkl": "swin_tiny_patch4_window7_224.pkl",
        "url": "https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_tiny_patch4_window7_224.pth",
        "embed_dim": 96,
        "depths": [2, 2, 6, 2],
        "num_heads": [3, 6, 12, 24],
        "window_size": 7,
        "pretrain_img_size": 224,
        "num_queries": 100,
    },
    "swin_small": {
        "tag": "swin_s",
        "pth": "swin_small_patch4_window7_224.pth",
        "pkl": "swin_small_patch4_window7_224.pkl",
        "url": "https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_small_patch4_window7_224.pth",
        "embed_dim": 96,
        "depths": [2, 2, 18, 2],
        "num_heads": [3, 6, 12, 24],
        "window_size": 7,
        "pretrain_img_size": 224,
        "num_queries": 100,
    },
    "swin_large": {
        "tag": "swin_l",
        "pth": "swin_large_patch4_window12_384_22k.pth",
        "pkl": "swin_large_patch4_window12_384_22k.pkl",
        "url": "https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_large_patch4_window12_384_22k.pth",
        "embed_dim": 192,
        "depths": [2, 2, 18, 2],
        "num_heads": [6, 12, 24, 48],
        "window_size": 12,
        "pretrain_img_size": 384,
        "num_queries": 200,
    },
}

opt = BACKBONE_OPTIONS[BACKBONE_VARIANT]
Path("ckpt").mkdir(exist_ok=True)
pth_path = Path("ckpt") / opt["pth"]
pkl_path = Path("ckpt") / opt["pkl"]

print("Backbone variant:", BACKBONE_VARIANT)
print("Downloading:", opt["url"])
!wget -q -nc -O "{pth_path}" "{opt['url']}"

if not pkl_path.exists():
    !python tools/convert-pretrained-swin-model-to-d2.py "{pth_path}" "{pkl_path}"
else:
    print("Converted checkpoint already exists:", pkl_path)

for p in [pth_path, pkl_path]:
    size_mb = p.stat().st_size / 1024 / 1024
    print(p, round(size_mb, 2), "MB")
    assert p.stat().st_size > 1_000_000, f"{p} looks too small; download/conversion probably failed."


In [ ]:
# Cell 6 - Create Swin backbone config and no-TTA train/eval scripts

%cd /content/ECLIPSE_STRONG_BACKBONE

from pathlib import Path
import textwrap

try:
    BACKBONE_VARIANT
    BACKBONE_OPTIONS
except NameError:
    BACKBONE_VARIANT = "swin_tiny"
    BACKBONE_OPTIONS = {
        "swin_tiny": {
            "tag": "swin_t",
            "pkl": "swin_tiny_patch4_window7_224.pkl",
            "embed_dim": 96,
            "depths": [2, 2, 6, 2],
            "num_heads": [3, 6, 12, 24],
            "window_size": 7,
            "pretrain_img_size": 224,
            "num_queries": 100,
        }
    }

opt = BACKBONE_OPTIONS[BACKBONE_VARIANT]
BACKBONE_TAG = opt["tag"]
DATA_ROOT = "/content/ECLIPSE_STRONG_BACKBONE/datasets"
RESULT_ROOT = f"results/ade_ps_{BACKBONE_TAG}"
BASE_STEP_CKPT = f"results/ade_ps_{BACKBONE_TAG}_100_step0.pth"
LOG_DIR = "/content/drive/MyDrive/ECLIPSE_STRONG_BACKBONE_LOGS"
Path(LOG_DIR).mkdir(parents=True, exist_ok=True)

# Stronger Colab A100 schedule.
# The original full ECLIPSE schedule can take days. This keeps the base step
# robust enough to be meaningful while staying below the original paper budget.
IMS_PER_BATCH = 8
BASE_MAX_ITER = 75000
PROMPT_100_50_MAX_ITER = 12000
PROMPT_100_10_MAX_ITER = 4000
PROMPT_100_5_MAX_ITER = 3000

# Evaluate only at the end of each training segment to avoid wasting hours on
# repeated validation passes. The final eval cells still produce the graph logs.
BASE_EVAL_PERIOD = BASE_MAX_ITER
BASE_CHECKPOINT_PERIOD = BASE_MAX_ITER
PROMPT_100_50_EVAL_PERIOD = PROMPT_100_50_MAX_ITER
PROMPT_100_10_EVAL_PERIOD = PROMPT_100_10_MAX_ITER
PROMPT_100_5_EVAL_PERIOD = PROMPT_100_5_MAX_ITER

print("Stronger Colab schedule:")
print(f"  base step: {BASE_MAX_ITER:,} iterations")
print(f"  100-50:    {PROMPT_100_50_MAX_ITER:,} iterations")
print(f"  100-10:    {PROMPT_100_10_MAX_ITER:,} iterations per step")
print(f"  100-5:     {PROMPT_100_5_MAX_ITER:,} iterations per step")

config_dir = Path("configs/ade20k/panoptic-segmentation/swin")
config_dir.mkdir(parents=True, exist_ok=True)
CFG_FILE = str(config_dir / f"maskformer2_{BACKBONE_TAG}_bs16_160k.yaml")

swin_config = f'''_BASE_: ../maskformer2_R50_bs16_160k.yaml
MODEL:
  BACKBONE:
    NAME: "D2SwinTransformer"
  SWIN:
    PRETRAIN_IMG_SIZE: {opt["pretrain_img_size"]}
    EMBED_DIM: {opt["embed_dim"]}
    DEPTHS: {opt["depths"]}
    NUM_HEADS: {opt["num_heads"]}
    WINDOW_SIZE: {opt["window_size"]}
    APE: False
    DROP_PATH_RATE: 0.3
    PATCH_NORM: True
  WEIGHTS: "ckpt/{opt["pkl"]}"
  PIXEL_MEAN: [123.675, 116.280, 103.530]
  PIXEL_STD: [58.395, 57.120, 57.375]
  MASK_FORMER:
    NUM_OBJECT_QUERIES: {opt["num_queries"]}
TEST:
  AUG:
    ENABLED: False
'''
Path(CFG_FILE).write_text(swin_config)
print("Wrote config:", CFG_FILE)
print(Path(CFG_FILE).read_text())

script_dir = Path(f"script/ade_ps_{BACKBONE_TAG}")
script_dir.mkdir(parents=True, exist_ok=True)

common_header = r'''#!/bin/bash
set -e

export DETECTRON2_DATASETS="__DATA_ROOT__"
ngpus=$(nvidia-smi --list-gpus | wc -l)

cfg_file="__CFG_FILE__"
base="__RESULT_ROOT__"
base_step_ckpt="__BASE_STEP_CKPT__"

mkdir -p "${base}"
cont_args=""
dist_args=""

meth_args="MODEL.MASK_FORMER.TEST.MASK_BG False MODEL.MASK_FORMER.PER_PIXEL False MODEL.MASK_FORMER.FOCAL True TEST.AUG.ENABLED False SOLVER.IMS_PER_BATCH __IMS_PER_BATCH__"

base_queries=__NUM_QUERIES__
dice_weight=5.0
mask_weight=5.0
soft_mask=False
soft_cls=False
deep_cls=True
'''
common_header = (common_header
    .replace("__DATA_ROOT__", DATA_ROOT)
    .replace("__CFG_FILE__", CFG_FILE)
    .replace("__RESULT_ROOT__", RESULT_ROOT)
    .replace("__BASE_STEP_CKPT__", BASE_STEP_CKPT)
    .replace("__IMS_PER_BATCH__", str(IMS_PER_BATCH))
    .replace("__NUM_QUERIES__", str(opt["num_queries"]))
)

base_script = common_header + r'''
step_args="CONT.BASE_CLS 100 CONT.INC_CLS 50 CONT.MODE overlap SEED 42"
class_weight=2.0
base_lr=0.0001
iter=__BASE_MAX_ITER__
num_prompts=0
exp_name="adps___TAG___base100"

weight_args="MODEL.MASK_FORMER.NUM_OBJECT_QUERIES ${base_queries} MODEL.MASK_FORMER.DICE_WEIGHT ${dice_weight} MODEL.MASK_FORMER.MASK_WEIGHT ${mask_weight} MODEL.MASK_FORMER.CLASS_WEIGHT ${class_weight} MODEL.MASK_FORMER.SOFTMASK ${soft_mask} CONT.SOFTCLS ${soft_cls} CONT.NUM_PROMPTS ${num_prompts} CONT.DEEP_CLS ${deep_cls}"
comm_args="OUTPUT_DIR ${base} ${meth_args} ${step_args} ${weight_args}"
inc_args="CONT.TASK 0 SOLVER.BASE_LR ${base_lr} TEST.EVAL_PERIOD ${iter} SOLVER.CHECKPOINT_PERIOD ${iter} SOLVER.MAX_ITER ${iter}"

if [ -f "${base_step_ckpt}" ]; then
  echo "Base checkpoint already exists: ${base_step_ckpt}"
  exit 0
fi

python train_inc.py --num-gpus ${ngpus} --config-file ${cfg_file} ${comm_args} ${inc_args} NAME ${exp_name} WANDB False

base_output="${base}/mya-pan_100-50-ov/${exp_name}/step0/model_final.pth"
if [ ! -f "${base_output}" ]; then
  echo "Expected base output not found: ${base_output}"
  exit 1
fi
cp "${base_output}" "${base_step_ckpt}"
echo "Saved shared 100-class base checkpoint to ${base_step_ckpt}"
'''
base_script = base_script.replace("__BASE_MAX_ITER__", str(BASE_MAX_ITER)).replace("__TAG__", BACKBONE_TAG)


def scenario_script(suffix, inc_cls, final_task, prompt_count, train_iter, train_deltas, eval_deltas, eval_period):
    task = f"mya-pan_100-{inc_cls}-ov"
    loop = ""
    if final_task > 1:
        loop_values = " ".join(str(t) for t in range(2, final_task + 1))
        loop = f'''
for t in {loop_values}; do
  inc_args="CONT.TASK ${{t}} SOLVER.MAX_ITER ${{iter}} SOLVER.BASE_LR ${{base_lr}} TEST.EVAL_PERIOD {eval_period} SOLVER.CHECKPOINT_PERIOD 500000"
  python train_inc.py --num-gpus ${{ngpus}} --config-file ${{cfg_file}} ${{comm_args}} ${{inc_args}} ${{cont_args}} ${{dist_args}} ${{vpt_args}} NAME ${{exp_name}} WANDB False
done
'''
    return common_header + f'''
step_args="CONT.BASE_CLS 100 CONT.INC_CLS {inc_cls} CONT.MODE overlap SEED 42"
exp_name="adps_{BACKBONE_TAG}_{suffix}"
task="{task}"
final_ckpt="${{base}}/${{task}}/${{exp_name}}/step{final_task}/model_final.pth"
exported_final="results/ade_ps_{BACKBONE_TAG}_{suffix}_final.pth"

if [ ! -f "${{base_step_ckpt}}" ]; then
  echo "Missing shared base checkpoint: ${{base_step_ckpt}}"
  echo "Run train_base_100.sh first."
  exit 1
fi

num_prompts={prompt_count}
iter={train_iter}
base_lr=0.0005
class_weight=10.0

backbone_freeze=True
trans_decoder_freeze=True
pixel_decoder_freeze=True
cls_head_freeze=True
mask_head_freeze=True
query_embed_freeze=True
prompt_deep=True
prompt_mask_mlp=True
prompt_no_obj_mlp=False
deltas={train_deltas}

weight_args="MODEL.MASK_FORMER.NUM_OBJECT_QUERIES ${{base_queries}} MODEL.MASK_FORMER.DICE_WEIGHT ${{dice_weight}} MODEL.MASK_FORMER.MASK_WEIGHT ${{mask_weight}} MODEL.MASK_FORMER.CLASS_WEIGHT ${{class_weight}} MODEL.MASK_FORMER.SOFTMASK ${{soft_mask}} CONT.SOFTCLS ${{soft_cls}} CONT.NUM_PROMPTS ${{num_prompts}}"
comm_args="OUTPUT_DIR ${{base}} ${{meth_args}} ${{step_args}} ${{weight_args}}"
vpt_args="CONT.BACKBONE_FREEZE ${{backbone_freeze}} CONT.CLS_HEAD_FREEZE ${{cls_head_freeze}} CONT.MASK_HEAD_FREEZE ${{mask_head_freeze}} CONT.PIXEL_DECODER_FREEZE ${{pixel_decoder_freeze}} CONT.QUERY_EMBED_FREEZE ${{query_embed_freeze}} CONT.TRANS_DECODER_FREEZE ${{trans_decoder_freeze}} CONT.PROMPT_MASK_MLP ${{prompt_mask_mlp}} CONT.PROMPT_NO_OBJ_MLP ${{prompt_no_obj_mlp}} CONT.PROMPT_DEEP ${{prompt_deep}} CONT.DEEP_CLS ${{deep_cls}} CONT.LOGIT_MANI_DELTAS ${{deltas}}"

if [ -f "${{final_ckpt}}" ]; then
  echo "Final prompt checkpoint already exists, skipping training: ${{final_ckpt}}"
else
  inc_args="CONT.TASK 1 SOLVER.MAX_ITER ${{iter}} SOLVER.BASE_LR ${{base_lr}} TEST.EVAL_PERIOD {eval_period} SOLVER.CHECKPOINT_PERIOD 500000 CONT.WEIGHTS ${{base_step_ckpt}}"
  python train_inc.py --num-gpus ${{ngpus}} --config-file ${{cfg_file}} ${{comm_args}} ${{inc_args}} ${{cont_args}} ${{dist_args}} ${{vpt_args}} NAME ${{exp_name}} WANDB False
{loop}
fi

if [ ! -f "${{final_ckpt}}" ]; then
  echo "Expected final checkpoint not found: ${{final_ckpt}}"
  exit 1
fi
cp "${{final_ckpt}}" "${{exported_final}}"

deltas={eval_deltas}
vpt_args="CONT.BACKBONE_FREEZE ${{backbone_freeze}} CONT.CLS_HEAD_FREEZE ${{cls_head_freeze}} CONT.MASK_HEAD_FREEZE ${{mask_head_freeze}} CONT.PIXEL_DECODER_FREEZE ${{pixel_decoder_freeze}} CONT.QUERY_EMBED_FREEZE ${{query_embed_freeze}} CONT.TRANS_DECODER_FREEZE ${{trans_decoder_freeze}} CONT.PROMPT_MASK_MLP ${{prompt_mask_mlp}} CONT.PROMPT_NO_OBJ_MLP ${{prompt_no_obj_mlp}} CONT.PROMPT_DEEP ${{prompt_deep}} CONT.DEEP_CLS ${{deep_cls}} CONT.LOGIT_MANI_DELTAS ${{deltas}}"
inc_args="CONT.TASK {final_task} CONT.WEIGHTS ${{final_ckpt}}"
python train_inc.py --eval-only --num-gpus ${{ngpus}} --config-file ${{cfg_file}} ${{comm_args}} ${{inc_args}} ${{cont_args}} ${{dist_args}} ${{vpt_args}} NAME ${{exp_name}} WANDB False
'''

scripts = {
    "train_base_100.sh": base_script,
    "train_eval_100_50.sh": scenario_script("100_50", 50, 1, 50, PROMPT_100_50_MAX_ITER, "[0.6,-0.6]", "[-0.4,-0.6]", PROMPT_100_50_EVAL_PERIOD),
    "train_eval_100_10.sh": scenario_script("100_10", 10, 5, 10, PROMPT_100_10_MAX_ITER, "[0.4,0.5,0.5,0.5,0.5,0.5]", "[0.4,0.5,0.5,0.5,0.5,0.5]", PROMPT_100_10_EVAL_PERIOD),
    "train_eval_100_5.sh": scenario_script("100_5", 5, 10, 10, PROMPT_100_5_MAX_ITER, "0.5", "[0.4,0.6,0.6,0.6,0.6,0.6,0.6,0.6,0.6,0.6,0.6]", PROMPT_100_5_EVAL_PERIOD),
}

for name, body in scripts.items():
    path = script_dir / name
    path.write_text(body)
    path.chmod(0o755)
    print("Wrote", path)

print("\nPreview base script:")
!sed -n '1,180p' "{script_dir / 'train_base_100.sh'}"


In [ ]:
# Cell 7 — Build ECLIPSE custom CUDA ops

%cd /content/ECLIPSE_STRONG_BACKBONE/mask2former/modeling/pixel_decoder/ops

!rm -rf build
!find . -name "*.so" -delete

# Patch old PyTorch API calls.
!sed -i 's/value.type()/value.scalar_type()/g' src/cuda/ms_deform_attn_cuda.cu
!sed -i 's/\.scalar_type()\.is_cuda()/.is_cuda()/g' src/cuda/ms_deform_attn_cuda.cu

!grep -n "AT_DISPATCH\|scalar_type\|is_cuda" src/cuda/ms_deform_attn_cuda.cu | head -30

!sh make.sh

!find . -name "*.so" -print

%cd /content/ECLIPSE_STRONG_BACKBONE

%env PYTHONPATH=/content/detectron2_src:/content/ECLIPSE_STRONG_BACKBONE:/content/ECLIPSE_STRONG_BACKBONE/mask2former/modeling/pixel_decoder/ops

import sys
import torch

print("Python:", sys.executable)
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)

# Make notebook kernel see Detectron2 source
if "/content/detectron2_src" not in sys.path:
    sys.path.insert(0, "/content/detectron2_src")

if "/content/ECLIPSE_STRONG_BACKBONE/mask2former/modeling/pixel_decoder/ops" not in sys.path:
    sys.path.insert(0, "/content/ECLIPSE_STRONG_BACKBONE/mask2former/modeling/pixel_decoder/ops")

from detectron2 import _C
print("Detectron2 _C OK")

import MultiScaleDeformableAttention
print("ECLIPSE MultiScaleDeformableAttention OK")


In [ ]:
# Fix/check ECLIPSE custom CUDA op visibility

%cd /content/ECLIPSE_STRONG_BACKBONE/mask2former/modeling/pixel_decoder/ops

from pathlib import Path
import sys

# Find the compiled .so file
so_files = list(Path(".").rglob("MultiScaleDeformableAttention*.so"))

print("Found .so files:")
for so in so_files:
    print(so.resolve())

if not so_files:
    raise FileNotFoundError("No MultiScaleDeformableAttention .so file found. The CUDA op did not compile.")

# Add the folder containing the .so to sys.path
so_dir = str(so_files[0].resolve().parent)

if so_dir not in sys.path:
    sys.path.insert(0, so_dir)

print("Added to sys.path:", so_dir)

import MultiScaleDeformableAttention
print("ECLIPSE MultiScaleDeformableAttention OK")


In [ ]:
# Cell 8 - Sanity check before training

%cd /content/ECLIPSE_STRONG_BACKBONE

from pathlib import Path
import sys
import os
import torch

try:
    BACKBONE_VARIANT
    BACKBONE_OPTIONS
except NameError:
    BACKBONE_VARIANT = "swin_tiny"
    BACKBONE_OPTIONS = {
        "swin_tiny": {"tag": "swin_t", "pkl": "swin_tiny_patch4_window7_224.pkl"}
    }

opt = BACKBONE_OPTIONS[BACKBONE_VARIANT]
backbone_tag = opt["tag"]

required = [
    Path("train_inc.py"),
    Path("datasets/ADEChallengeData2016"),
    Path("ckpt") / opt["pkl"],
    Path(f"configs/ade20k/panoptic-segmentation/swin/maskformer2_{backbone_tag}_bs16_160k.yaml"),
    Path(f"script/ade_ps_{backbone_tag}/train_base_100.sh"),
    Path(f"script/ade_ps_{backbone_tag}/train_eval_100_50.sh"),
    Path(f"script/ade_ps_{backbone_tag}/train_eval_100_10.sh"),
    Path(f"script/ade_ps_{backbone_tag}/train_eval_100_5.sh"),
]

missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing files/folders:\n" + "\n".join(missing))

print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ops_root = Path("/content/ECLIPSE_STRONG_BACKBONE/mask2former/modeling/pixel_decoder/ops")
so_files = list(ops_root.rglob("MultiScaleDeformableAttention*.so"))
print("\nFound ECLIPSE custom op files:")
for so in so_files:
    print(so.resolve())
if not so_files:
    raise FileNotFoundError("MultiScaleDeformableAttention .so was not found. Rerun Cell 7.")

so_dir = str(so_files[0].resolve().parent)
paths_to_add = ["/content/detectron2_src", "/content/ECLIPSE_STRONG_BACKBONE", so_dir]
for path in paths_to_add:
    if path not in sys.path:
        sys.path.insert(0, path)
os.environ["PYTHONPATH"] = ":".join(paths_to_add + [os.environ.get("PYTHONPATH", "")])

import detectron2
from detectron2 import _C
print("\nDetectron2 _C OK")

import MultiScaleDeformableAttention
print("ECLIPSE MultiScaleDeformableAttention OK")
print("\nNo-TTA stronger-backbone training setup is ready.")


In [ ]:
# Cell 8.5 - Restore/save Swin-T training checkpoints through Google Drive

%cd /content/ECLIPSE_STRONG_BACKBONE

from pathlib import Path
import shutil

try:
    BACKBONE_VARIANT
    BACKBONE_OPTIONS
except NameError:
    BACKBONE_VARIANT = "swin_tiny"
    BACKBONE_OPTIONS = {"swin_tiny": {"tag": "swin_t"}}

backbone_tag = BACKBONE_OPTIONS[BACKBONE_VARIANT]["tag"]
CHECKPOINT_BACKUP_DIR = Path("/content/drive/MyDrive/ECLIPSE_STRONG_BACKBONE_CHECKPOINTS") / backbone_tag
CHECKPOINT_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

RESULT_ROOT = Path(f"results/ade_ps_{backbone_tag}")
CHECKPOINT_MANIFEST = {
    "base100": [
        Path(f"results/ade_ps_{backbone_tag}_100_step0.pth"),
        RESULT_ROOT / "mya-pan_100-50-ov" / f"adps_{backbone_tag}_base100" / "step0" / "model_final.pth",
    ],
    "100_50_final": [
        Path(f"results/ade_ps_{backbone_tag}_100_50_final.pth"),
        RESULT_ROOT / "mya-pan_100-50-ov" / f"adps_{backbone_tag}_100_50" / "step1" / "model_final.pth",
    ],
    "100_10_final": [
        Path(f"results/ade_ps_{backbone_tag}_100_10_final.pth"),
        RESULT_ROOT / "mya-pan_100-10-ov" / f"adps_{backbone_tag}_100_10" / "step5" / "model_final.pth",
    ],
    "100_5_final": [
        Path(f"results/ade_ps_{backbone_tag}_100_5_final.pth"),
        RESULT_ROOT / "mya-pan_100-5-ov" / f"adps_{backbone_tag}_100_5" / "step10" / "model_final.pth",
    ],
}

def restore_swin_training_checkpoints():
    restored = []
    for name, local_paths in CHECKPOINT_MANIFEST.items():
        backup = CHECKPOINT_BACKUP_DIR / f"{name}.pth"
        if not backup.exists():
            continue
        for local_path in local_paths:
            local_path.parent.mkdir(parents=True, exist_ok=True)
            if not local_path.exists():
                shutil.copy2(backup, local_path)
                restored.append(str(local_path))
    if restored:
        print("Restored checkpoints from Drive:")
        for item in restored:
            print("  ", item)
    else:
        print("No Drive checkpoint backups found to restore yet.")

def save_swin_training_checkpoints():
    saved = []
    for name, local_paths in CHECKPOINT_MANIFEST.items():
        source = next((path for path in local_paths if path.exists()), None)
        if source is None:
            continue
        backup = CHECKPOINT_BACKUP_DIR / f"{name}.pth"
        shutil.copy2(source, backup)
        saved.append((str(source), str(backup)))
    if saved:
        print("Saved checkpoints to Drive:")
        for source, backup in saved:
            print(f"  {source} -> {backup}")
    else:
        print("No local checkpoints found to save yet.")

restore_swin_training_checkpoints()
print("Checkpoint backup folder:", CHECKPOINT_BACKUP_DIR)
print("Training cells below call save_swin_training_checkpoints() after each completed stage.")


In [ ]:
# Cell 9 - Train the shared 100-class Swin-T base step once

%cd /content/ECLIPSE_STRONG_BACKBONE

from pathlib import Path
import os

backbone_tag = BACKBONE_OPTIONS[BACKBONE_VARIANT]["tag"]
ops_root = Path("/content/ECLIPSE_STRONG_BACKBONE/mask2former/modeling/pixel_decoder/ops")
so_files = list(ops_root.rglob("MultiScaleDeformableAttention*.so"))
if not so_files:
    raise FileNotFoundError("MultiScaleDeformableAttention .so was not found. Rerun Cell 7.")
so_dir = str(so_files[0].resolve().parent)
pythonpath = f"/content/detectron2_src:/content/ECLIPSE_STRONG_BACKBONE:{so_dir}:{os.environ.get('PYTHONPATH', '')}"

!mkdir -p /content/drive/MyDrive/ECLIPSE_STRONG_BACKBONE_LOGS
!PYTHONPATH="{pythonpath}" bash "script/ade_ps_{backbone_tag}/train_base_100.sh" 2>&1 | tee "output_{backbone_tag}_base100.txt"
!cp "output_{backbone_tag}_base100.txt" /content/drive/MyDrive/ECLIPSE_STRONG_BACKBONE_LOGS/

try:
    save_swin_training_checkpoints()
except NameError:
    print("Run Cell 8.5 to enable Google Drive checkpoint saving/restoring.")

print("Finished or reused shared Swin base checkpoint.")


In [ ]:
# Cell 10 - Train and evaluate Swin-T 100-50, no TTA

%cd /content/ECLIPSE_STRONG_BACKBONE

from pathlib import Path
import os

backbone_tag = BACKBONE_OPTIONS[BACKBONE_VARIANT]["tag"]
ops_root = Path("/content/ECLIPSE_STRONG_BACKBONE/mask2former/modeling/pixel_decoder/ops")
so_files = list(ops_root.rglob("MultiScaleDeformableAttention*.so"))
if not so_files:
    raise FileNotFoundError("MultiScaleDeformableAttention .so was not found. Rerun Cell 7.")
so_dir = str(so_files[0].resolve().parent)
pythonpath = f"/content/detectron2_src:/content/ECLIPSE_STRONG_BACKBONE:{so_dir}:{os.environ.get('PYTHONPATH', '')}"

!mkdir -p /content/drive/MyDrive/ECLIPSE_STRONG_BACKBONE_LOGS
!PYTHONPATH="{pythonpath}" bash "script/ade_ps_{backbone_tag}/train_eval_100_50.sh" 2>&1 | tee "output_{backbone_tag}_100_50.txt"
!cp "output_{backbone_tag}_100_50.txt" /content/drive/MyDrive/ECLIPSE_STRONG_BACKBONE_LOGS/

try:
    save_swin_training_checkpoints()
except NameError:
    print("Run Cell 8.5 to enable Google Drive checkpoint saving/restoring.")

print("Finished Swin 100-50 training/evaluation.")


In [ ]:
# Cell 11 - Check 100-50 all-PQ

%cd /content/ECLIPSE_STRONG_BACKBONE
backbone_tag = BACKBONE_OPTIONS[BACKBONE_VARIANT]["tag"]
!grep -A 3 -i "copypaste: PQ" "output_{backbone_tag}_100_50.txt" || true


In [ ]:
# Cell 12 - Train and evaluate Swin-T 100-10, no TTA

%cd /content/ECLIPSE_STRONG_BACKBONE

from pathlib import Path
import os

backbone_tag = BACKBONE_OPTIONS[BACKBONE_VARIANT]["tag"]
ops_root = Path("/content/ECLIPSE_STRONG_BACKBONE/mask2former/modeling/pixel_decoder/ops")
so_files = list(ops_root.rglob("MultiScaleDeformableAttention*.so"))
if not so_files:
    raise FileNotFoundError("MultiScaleDeformableAttention .so was not found. Rerun Cell 7.")
so_dir = str(so_files[0].resolve().parent)
pythonpath = f"/content/detectron2_src:/content/ECLIPSE_STRONG_BACKBONE:{so_dir}:{os.environ.get('PYTHONPATH', '')}"

!mkdir -p /content/drive/MyDrive/ECLIPSE_STRONG_BACKBONE_LOGS
!PYTHONPATH="{pythonpath}" bash "script/ade_ps_{backbone_tag}/train_eval_100_10.sh" 2>&1 | tee "output_{backbone_tag}_100_10.txt"
!cp "output_{backbone_tag}_100_10.txt" /content/drive/MyDrive/ECLIPSE_STRONG_BACKBONE_LOGS/

try:
    save_swin_training_checkpoints()
except NameError:
    print("Run Cell 8.5 to enable Google Drive checkpoint saving/restoring.")

print("Finished Swin 100-10 training/evaluation.")


In [ ]:
# Cell 13 - Train and evaluate Swin-T 100-5, no TTA

%cd /content/ECLIPSE_STRONG_BACKBONE

from pathlib import Path
import os

backbone_tag = BACKBONE_OPTIONS[BACKBONE_VARIANT]["tag"]
ops_root = Path("/content/ECLIPSE_STRONG_BACKBONE/mask2former/modeling/pixel_decoder/ops")
so_files = list(ops_root.rglob("MultiScaleDeformableAttention*.so"))
if not so_files:
    raise FileNotFoundError("MultiScaleDeformableAttention .so was not found. Rerun Cell 7.")
so_dir = str(so_files[0].resolve().parent)
pythonpath = f"/content/detectron2_src:/content/ECLIPSE_STRONG_BACKBONE:{so_dir}:{os.environ.get('PYTHONPATH', '')}"

!mkdir -p /content/drive/MyDrive/ECLIPSE_STRONG_BACKBONE_LOGS
!PYTHONPATH="{pythonpath}" bash "script/ade_ps_{backbone_tag}/train_eval_100_5.sh" 2>&1 | tee "output_{backbone_tag}_100_5.txt"
!cp "output_{backbone_tag}_100_5.txt" /content/drive/MyDrive/ECLIPSE_STRONG_BACKBONE_LOGS/

try:
    save_swin_training_checkpoints()
except NameError:
    print("Run Cell 8.5 to enable Google Drive checkpoint saving/restoring.")

print("Finished Swin 100-5 training/evaluation.")


In [ ]:
# Cell 14 - Extract final all-PQ lines for all Swin-T scenarios

%cd /content/ECLIPSE_STRONG_BACKBONE

backbone_tag = BACKBONE_OPTIONS[BACKBONE_VARIANT]["tag"]

print("===== 100-50 =====")
!grep -A 3 -i "copypaste: PQ" "output_{backbone_tag}_100_50.txt" || true

print("\n===== 100-10 =====")
!grep -A 3 -i "copypaste: PQ" "output_{backbone_tag}_100_10.txt" || true

print("\n===== 100-5 =====")
!grep -A 3 -i "copypaste: PQ" "output_{backbone_tag}_100_5.txt" || true


In [ ]:
# Cell 15 - Graph article results vs final stronger-backbone results

%cd /content/ECLIPSE_STRONG_BACKBONE

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import re

backbone_tag = BACKBONE_OPTIONS[BACKBONE_VARIANT]["tag"]
backbone_label = BACKBONE_VARIANT.replace("_", "-")

scenarios = ['100-5\n(11 Tasks)', '100-10\n(6 Tasks)', '100-50\n(2 Tasks)']
article_scores = [32.9, 33.9, 35.6]
log_files = [
    Path(f'output_{backbone_tag}_100_5.txt'),
    Path(f'output_{backbone_tag}_100_10.txt'),
    Path(f'output_{backbone_tag}_100_50.txt'),
]

def parse_all_pq(log_path):
    if not log_path.exists():
        raise FileNotFoundError(f"Missing {log_path}. Run the corresponding train/eval cell first.")
    lines = log_path.read_text(errors='ignore').splitlines()
    for idx, line in enumerate(lines):
        if 'copypaste:' in line.lower() and re.search(r'\bPQ\b', line):
            for candidate in lines[idx + 1:idx + 6]:
                if 'copypaste:' not in candidate.lower():
                    continue
                numbers = re.findall(r'[-+]?\d+(?:\.\d+)?', candidate)
                if numbers:
                    return float(numbers[0])
    raise ValueError(f"Could not parse all-PQ from {log_path}")

strong_backbone_scores = [parse_all_pq(p) for p in log_files]

x = np.arange(len(scenarios))
width = 0.36

fig, ax = plt.subplots(figsize=(10, 6))
r1 = ax.bar(x - width / 2, article_scores, width, label='Original article')
r2 = ax.bar(x + width / 2, strong_backbone_scores, width, label=f'{backbone_label} 75k-base schedule, no TTA')

ax.set_ylabel('All PQ')
ax.set_title('ECLIPSE ADE20K-Panoptic: Original vs Swin-T 75k-Base Run')
ax.set_xticks(x)
ax.set_xticklabels(scenarios)
ax.set_ylim(30, max(article_scores + strong_backbone_scores) + 2)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.45)
ax.bar_label(r1, padding=3, fmt='%.2f')
ax.bar_label(r2, padding=3, fmt='%.2f')

plt.tight_layout()
out_dir = Path('/content/drive/MyDrive/ECLIPSE_STRONG_BACKBONE_LOGS')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f'eclipse_{backbone_tag}_no_tta_article_comparison.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()

print('Original article all-PQ:', article_scores)
print(f'{backbone_label} no-TTA all-PQ:', strong_backbone_scores)
print('Saved graph to:', out_path)


## Report wording

Use this after the notebook finishes:

> We reproduced the ADE20K-Panoptic continual segmentation setup from ECLIPSE and changed only the frozen-base-model backbone path. Instead of the original ResNet-50 backbone, we trained a new 100-class base step with a Swin-T backbone initialized from the official ImageNet checkpoint using a stronger 75k-base Colab schedule, then froze the base model and trained the continual prompt steps for 100-50, 100-10, and 100-5. Evaluation used the normal single-scale inference path with no test-time augmentation. We compare the final all-PQ values from the new logs against the original article values.

Important: the graph parses the final **all PQ** value from each fresh Swin-T evaluation log. This notebook uses a 75k-base Swin-T schedule plus Google Drive checkpoint backup/restore; report it as a 75k-base Swin-T run unless you manually restore the full paper-length iteration counts.
